In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

local_mount = "/Users/snaranjo/Desktop/neurotranslate/mount_point"
janine_group_path = "/ceph/chpc/shared/janine_bijsterbosch_group"
chpc_data_path = janine_group_path+"/l.lexi_scratch/WAPIAW2024/Data/ABCD"
full_path = local_mount + chpc_data_path
get_df_ABCD_siteconfound = pd.read_csv(full_path + "/confounds_nodum.csv")
# print(get_df_ABCD_siteconfound)

get_only_ID = get_df_ABCD_siteconfound[["Session","site"]]
print(get_only_ID.shape)

chpc_surf_path = janine_group_path+"/naranjorincon_scratch/NeuroTranslate/generate_ICA"
surf_path = local_mount + chpc_surf_path
get_ABCD_surface_ID = pd.read_csv(surf_path + "/ABCD_ICA_subjids.txt", header=None)
get_ABCD_surface_ID.rename( columns={0:'Session'}, inplace=True)
print(get_ABCD_surface_ID.shape)

get_matching_ABCD_ID = get_only_ID[get_only_ID['Session'].isin(get_ABCD_surface_ID['Session'])]
print(get_matching_ABCD_ID.shape)

print(f"Total num of different sites: {get_matching_ABCD_ID['site'].nunique()}")
site_subj_count = get_matching_ABCD_ID['site'].value_counts()
# print(site_subj_count)

# split into train and validate and test -  Based on these counts and what we know about sites 1 & 2
# train will be first 14 sites and remaining 6 are validation/test 3/3

define_validation = [15, 5, 7]
define_test = [18, 8, 12]
site_nums = np.arange(1,21,1)
mask = np.isin(site_nums, [define_validation, define_test])
define_train = site_nums[~mask] # 1-20 not in either validation or test
# print(define_train)
# define_train = site_nums

train_df = get_matching_ABCD_ID[get_matching_ABCD_ID['site'].isin(define_train)]
val_df = get_matching_ABCD_ID[get_matching_ABCD_ID['site'].isin(define_validation)]
test_df = get_matching_ABCD_ID[get_matching_ABCD_ID['site'].isin(define_test)]

print(f"TRAIN: {train_df} \nVALIDATION: {val_df} \nTEST:{test_df}")

path_to_save_split = local_mount + janine_group_path + "/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD/schaefer_mats/netmat_d100"
# get_matching_ABCD_ID.to_csv(path_to_save_split+'/ABCD_full_ID_site.csv', index=False)
# train_df.to_csv(path_to_save_split+'/ABCD_train_ID_site.csv', index=False)
# val_df.to_csv(path_to_save_split+'/ABCD_val_ID_site.csv', index=False)
# test_df.to_csv(path_to_save_split+'/ABCD_test_ID_site.csv', index=False)


(9714, 2)
(8906, 1)
(8673, 2)
Total num of different sites: 20
TRAIN:               Session  site
0     NDARINV003RTV85     6
1     NDARINV007W6H7B     1
3     NDARINV00CY2MDM    20
5     NDARINV00J52GPG    17
6     NDARINV00LH735Y     3
...               ...   ...
9709  NDARINVZUJYFZPW     2
9710  NDARINVZUN3P64X     1
9711  NDARINVZUXHPX3N    17
9712  NDARINVZV4EUGNG     9
9713  NDARINVZV54BWRE    20

[7121 rows x 2 columns] 
VALIDATION:               Session  site
2     NDARINV00BD7VDC     7
39    NDARINV03KMHMJJ     7
47    NDARINV04EUBGTM    15
54    NDARINV052HU3CU    15
78    NDARINV07RPB2TU     5
...               ...   ...
9679  NDARINVZPE7604M    15
9695  NDARINVZT7CGM7G     7
9705  NDARINVZU6F3PY8     5
9706  NDARINVZU8PLK49     7
9707  NDARINVZUDPGDWJ     5

[928 rows x 2 columns] 
TEST:              Session  site
4     NDARINV00HEV6HB    12
11    NDARINV00UMK5VC    12
13    NDARINV010ZM3H9    12
26    NDARINV028WCTG6    12
41    NDARINV03XVEBPM     8
...               ... 

## TRYING TO BREAK DOWN PREPROCESSING

In [52]:
import pandas as pd
import nibabel as nb
import numpy as np

# import yaml
import os
# import argparse

local_mount = "/Users/snaranjo/Desktop/neurotranslate/mount_point"
def write_to_file(content):
    with open(f"{local_mount}/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/surf2netmat/batch/SiT_prep.print", 'a') as file:
        file.write(str(content) + '\n')

#### PARAMETERS #####
ico = 6
sub_ico = 2

# configuration = config['data']['configuration']  # template #template #native
num_channels = 15
split = "test"
translation = "ICAd15_schfd100"
output_folder = local_mount+"/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD/ICA_maps/ICAd15_ico02"
dataset = "ABCD"
path_to_data = local_mount+"/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD/ICA_maps/ICAd15_ico06"
nm_fs_data = "d15_fs_LR"
sub_ids_path = local_mount+"/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/brain_reps_datasets/ABCD/schaefer_mats/netmat_d100"

num_vertices = 153
num_patches = 320
chosen_hemi = "1L"


In [53]:
if dataset == "HCPYA":
        ids = pd.read_csv(os.path.join(sub_ids_path, '{}/{}_upt.csv'.format(translation,split)))['ids'].to_numpy().reshape(-1)
elif dataset == "ABCD":
        ids = pd.read_csv(f"{sub_ids_path}/{split}_netmat.csv", header=None).iloc[0,:].to_numpy().reshape(-1)

print(ids)
print(ids.shape)

['NDARINV00HEV6HB' 'NDARINV00UMK5VC' 'NDARINV010ZM3H9' 'NDARINV028WCTG6'
 'NDARINV03XVEBPM' 'NDARINV04GAB2AA' 'NDARINV04P0G6LK' 'NDARINV05ATJ1V1'
 'NDARINV05LGG3GZ' 'NDARINV06U4DYFY' 'NDARINV07RAHHYH' 'NDARINV097LUBWX'
 'NDARINV0A6WVRZY' 'NDARINV0BVP2PTD' 'NDARINV0CV2Y4YR' 'NDARINV0DMRP60R'
 'NDARINV0DYF4WPG' 'NDARINV0F78WV5U' 'NDARINV0GWJ7GFF' 'NDARINV0HFHMCFZ'
 'NDARINV0HJGCR32' 'NDARINV0JGVD4F3' 'NDARINV0LDK94T8' 'NDARINV0MPBK7TU'
 'NDARINV0MRY4Z3E' 'NDARINV0MXJ7AZN' 'NDARINV0NGG5VLJ' 'NDARINV0NNE55L0'
 'NDARINV0U1JRFPK' 'NDARINV0V1TNU11' 'NDARINV0W4PR2WT' 'NDARINV0WHFCZKU'
 'NDARINV0WYCV590' 'NDARINV0XFVW4LF' 'NDARINV0XKVPBB0' 'NDARINV0XVGNCYR'
 'NDARINV0YWBYAGP' 'NDARINV0ZHHWCMZ' 'NDARINV0ZPPK2ML' 'NDARINV11ZVVFHK'
 'NDARINV13FP25D3' 'NDARINV14BPGL15' 'NDARINV14PHMUNU' 'NDARINV14PNN5H9'
 'NDARINV15M33G49' 'NDARINV18HXV64A' 'NDARINV18RUWAJN' 'NDARINV18YX7994'
 'NDARINV1925AD9X' 'NDARINV19GEFHUC' 'NDARINV1AJNN0V2' 'NDARINV1B9D3LNM'
 'NDARINV1BHHLZ52' 'NDARINV1D95ZF1L' 'NDARINV1E2PBK

/Users/snaranjo/miniconda3/envs/dahan_optimus/lib/python3.7/site-packages/IPython/core/interactiveshell.py:3457: DtypeWarning: Columns (0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,24

In [54]:
if dataset == "HCPYA":
    if split == 'train':
        # labels = pd.read_csv(os.path.join(label_path, '{}/{}.csv'.format(task,'vecmats'))).to_numpy()
        labels = pd.read_csv(os.path.join(sub_ids_path, '{}/{}.csv'.format(translation,'vecmats_train_upt'))).to_numpy()
    elif split == 'validation':
        # labels = pd.read_csv(os.path.join(label_path, '{}/{}.csv'.format(task,'vecmats_validation'))).to_numpy() 
        labels = pd.read_csv(os.path.join(sub_ids_path, '{}/{}.csv'.format(translation,'vecmats_validation_upt'))).to_numpy()  
    elif split == 'test':
        # labels = pd.read_csv(os.path.join(label_path, '{}/{}.csv'.format(task,'vecmats_test'))).to_numpy()
        labels = pd.read_csv(os.path.join(sub_ids_path, '{}/{}.csv'.format(translation,'vecmats_test_upt'))).to_numpy()    
elif dataset == "ABCD":
    labels = pd.read_csv(f"{sub_ids_path}/{split}_netmat.csv").to_numpy()

labels = labels.T #transpose to make subjXvector
labels_check_nans = np.isnan(labels).sum() #check if has nans and how many
print(f"Label NaN Count: {labels_check_nans}")

Label NaN Count: 0


In [55]:
if labels_check_nans !=0: # if there are nans in vecnetmats
    # when need to run local - add this to start: '/Users/snaranjo/Desktop/neurotranslate/mount_point'
    read_nan_subj_netmats = pd.read_csv('/Users/snaranjo/Desktop/neurotranslate/mount_point/ceph/chpc/shared/janine_bijsterbosch_group/naranjorincon_scratch/NeuroTranslate/surf2netmat/utils/subjects_with_nan.txt',sep=" ", header=None)
    read_nan_subj_netmats = read_nan_subj_netmats[0].values #makes df into numpy, not sure how differnt if at all from df.to_numpy()
    print(f"Subs VEC_NETMATs to remove - {read_nan_subj_netmats}")
    nan_mask = ~np.isin(ids, read_nan_subj_netmats) #ids is NDA* and read_nan_subj_netmats are those needed removed, so nan_mask is NOT is in, so ids excluding the ones we want to remove
    new_ids = ids[nan_mask] # here we set that as idx and get new_ids. If 3 subs to remove then new_ids = count(ids) - 3
    print(f"Full IDs no vecnetmat NaN -- {new_ids.shape}") # make sure this makes sense based on above print "read_nan_subj_netmats"

    nan_indices = np.where(np.isin(ids, read_nan_subj_netmats))[0]
    print(nan_indices)

    mask = np.ones(labels.shape[0], dtype=bool) # make mask of size of labels
    mask[nan_indices] = False # make where we found the NaNs ie subjects removed into false (all tohers are truw cause ones)
    labels_nan_clean = labels[mask] # update accordingly
    print(f"Some subjects had nans(N={len(nan_indices)}). Removed, new ids and labels shape ==> {labels_nan_clean.shape}; original:{labels.shape}")
else:
    labels_nan_clean = labels
    new_ids = ids

In [56]:
print("split choice:"+split)
ids = new_ids #ids with removed labels
num_subjects = ids.shape[0]
print('Num of subjects:'+str(num_subjects))
print('')

data = [] # list of numpy arrays each is a numpy array version of the shape.gii info
nan_surf_subject_count = 0
sub_ids_nan_vals = []
print(f'Dataset is {dataset}')
if chosen_hemi == '1L':
    print('Left hemisphere was chosen.')
    for i, id in enumerate(ids): # reads in actual id num with 'id' inside the pandas column from the read csv, see above ids variable

        if dataset == "ABCD":
            filename = os.path.join(f'{path_to_data}/resamp_{id}.L.shape.gii')
        else:
            filename = os.path.join(path_to_data,'ICA_fs_LR_32',nm_fs_data,'resamp_{}.L.shape.gii'.format(id)) # shape.gii is a metric file for values at each vertex, in our case the ICA values
        
        check_exists = os.path.exists(filename)
        if check_exists is False:
            print('This path did not exists, skipping:{}'.format(filename))
            continue

        data.append(np.array(nb.load(filename).agg_data())[:num_channels,:])

        if np.isnan(data[i]).sum() != 0: # so if there are nans
            data_check_nans = np.isnan(data[i]).sum()
            nan_surf_subject_count += 1
            sub_ids_nan_vals.append(id)
            print(f"DATA SUBJ ID:{id}/ iter:{i} NaN Count - {data_check_nans}")

        if i%200==0:
            print('\nLoading GIFTI for subject: {}'.format(i))
            print('\nActual stored data value, presumably aggregated through channels=inputdim:{}'.format(len(data[i])))

            

split choice:test
Num of subjects:624

Dataset is ABCD
Left hemisphere was chosen.

Loading GIFTI for subject: 0

Actual stored data value, presumably aggregated through channels=inputdim:15

Loading GIFTI for subject: 200

Actual stored data value, presumably aggregated through channels=inputdim:15

Loading GIFTI for subject: 400

Actual stored data value, presumably aggregated through channels=inputdim:15

Loading GIFTI for subject: 600

Actual stored data value, presumably aggregated through channels=inputdim:15


In [59]:
if sub_ids_nan_vals: # if True, then list is not empty == there are subjects with NaNs
    clean_subs = ~np.isin(ids, sub_ids_nan_vals)
    df_version = pd.DataFrame(ids[clean_subs])
    df_version.to_csv(f'{sub_ids_path}/{split}_subj_IDs_clean.csv')

    old_data = data
    data_nan_indices = np.where(np.isin(ids, sub_ids_nan_vals))[0]
    print(f"surf_nan_idx - {data_nan_indices}") # subjects with surf data that also have NaNs and need removal

    # merge data and labels cause both have different subjects with nans
    mask_data = np.ones(len(old_data), dtype=bool) # make mask of length of data == N, 
    print(f"Data mask shape: {mask_data.shape}")
    mask_data[data_nan_indices] = False # make where we found the NaNs ie subjects removed into false (all tohers are truw cause ones)
    data_np = np.asarray(old_data) # make list to np array
    data_nan_clean = data_np[mask_data] # that also allows us to index with the mask
    print(f"Clean data shape, should be N - surf_nan_idx: {data_nan_clean.shape}")
    
    # below is merging the two, so if surf needed subs removed then here we merge s.t. labels also now reflect the same and removed those subjs without surf
    mask_surfaces = np.ones(labels_nan_clean.shape[0], dtype=bool)
    mask_surfaces[data_nan_indices] = False # where the surf idx are for NaN make that false in label variable
    merged_labels_with_surf = labels_nan_clean[mask_surfaces] # now that those are false, index w mask and we have new labels which is clean of own NaNs and surf NaNs
    data = data_nan_clean
    labels = merged_labels_with_surf
    ids = ids[mask_surfaces]
    num_subjects = data.shape[0] # can be either data or labels, both should be equal anyway
else:
    clean_subs = ~np.isin(ids, sub_ids_nan_vals)
    df_version = pd.DataFrame(ids[clean_subs])
    df_version.to_csv(f'{sub_ids_path}/{split}_subj_IDs_clean.csv')
    print('CCLLEEAANN')
    data = np.asarray(data)

print(f"Cleaned surfaces and vec_netmats, make sure are same size and merged correctly: \nSURFACES:{data.shape} \nVEC_NETMATS:{labels.shape}")
print(f"Count of subjects that needed to be removed based on SURF NaNs: {len(sub_ids_nan_vals)}")

assert data.shape[0] == labels.shape[0], "surface IDs and netmat IDs are not equal"    
assert np.isnan(data).sum() == 0 and np.isnan(labels).sum() == 0, "Somehow cleaning failed try again."


CCLLEEAANN
Cleaned surfaces and vec_netmats, make sure are same size and merged correctly: 
SURFACES:(624, 15, 40962) 
VEC_NETMATS:(624, 4950)
Count of subjects that needed to be removed based on SURF NaNs: 0


In [45]:
ids2 = new_ids
data_nan_indices = np.where(np.isin(ids2, sub_ids_nan_vals))[0]
print(f"surf_nan_idx - {data_nan_indices}")
clean_subs = ~np.isin(ids2, sub_ids_nan_vals)
print(ids2[clean_subs])

df_version = pd.DataFrame(ids2[clean_subs])
df_version.to_csv(f'{sub_ids_path}/TEST.csv')

surf_nan_idx - [416 443 469 571 653 679 704]
['NDARINV00BD7VDC' 'NDARINV03KMHMJJ' 'NDARINV04EUBGTM' 'NDARINV052HU3CU'
 'NDARINV07RPB2TU' 'NDARINV08WDFUCE' 'NDARINV08YFFYY2' 'NDARINV09ZE6UUK'
 'NDARINV0A4ZDYNL' 'NDARINV0A86UD86' 'NDARINV0A87RKWD' 'NDARINV0AEBMADL'
 'NDARINV0CP9XGTP' 'NDARINV0FV544H9' 'NDARINV0JR6Y529' 'NDARINV0JVUB582'
 'NDARINV0JW2EU4V' 'NDARINV0L974J49' 'NDARINV0LA6MBBY' 'NDARINV0LEM88KP'
 'NDARINV0MCARYLD' 'NDARINV0ML93ZXT' 'NDARINV0N0JE94U' 'NDARINV0P34UPZ9'
 'NDARINV0PANNNLF' 'NDARINV0PJ81CA5' 'NDARINV0RL1BGTN' 'NDARINV0TEH16CM'
 'NDARINV0ULZKXXG' 'NDARINV0UV5WZUN' 'NDARINV0XTVAGV2' 'NDARINV101HGWPC'
 'NDARINV10DN9UHY' 'NDARINV12WR8W3U' 'NDARINV13NMXD4V' 'NDARINV13PRF0TM'
 'NDARINV174LD3GC' 'NDARINV1APPZYY8' 'NDARINV1BZ0TUT9' 'NDARINV1CXXN3K0'
 'NDARINV1E2WJC9K' 'NDARINV1FDC7YAJ' 'NDARINV1FGBFDT9' 'NDARINV1GYJNY83'
 'NDARINV1JGK90HZ' 'NDARINV1KHXW139' 'NDARINV1KWBW4RD' 'NDARINV1NJK6BVY'
 'NDARINV1R4KE1UN' 'NDARINV1R7RYN1P' 'NDARINV1U2B4E90' 'NDARINV1VNU5KTM'
 'NDAR

In [ ]:
# # check the nans
# # sub_ids_nan_vals = [271, 665]
# # ii = ids[sub_ids_nan_vals[0]]
# # ii = sub_ids_nan_vals[0]
# # filename = os.path.join(f'{local_mount}/{path_to_data}/resamp_{ii}.L.shape.gii')
# # print(f"resamp_{ii}.L.shape.gii")
# # check_agg_data = nb.load(filename).agg_data()

# # print(len(check_agg_data))
# # print(f"NaNa -- {np.isnan(check_agg_data).sum()}")

# # nan_mask = ~np.isin(ids, read_nan_subj_netmats)
# # new_ids = ids[nan_mask]
# # print(f"Full IDs no vecnetmat NaN -- {new_ids.shape}")
# # if sub_ids_nan_vals:
# #     data_nan_indices = np.where(np.isin(ids, sub_ids_nan_vals))[0]
# #     print(f"sur nan Idx - {data_nan_indices}")

# #     mask_data = np.ones(len(data), dtype=bool) # make mask of size of labels
# #     print(mask_data.shape)
# #     # print(len(data))
# #     mask_data[data_nan_indices] = False # make where we found the NaNs ie subjects removed into false (all tohers are truw cause ones)
# #     data = np.asarray(data)
# #     new_data = data[mask_data]
# #     data = new_data
# #     print(new_data.shape) # subjects removed that have NaNs. 
# #     print(new_label.shape)

# #     mask_surfaces = np.ones(new_label.shape[0], dtype=bool)
# #     mask_surfaces[data_nan_indices] = False
# #     new_new_label = new_label[mask_surfaces]
# #     print(new_new_label.shape)

# labels_nan_clean=new_label
# print(sub_ids_nan_vals)
# if sub_ids_nan_vals: # if True, then list is not empty == there are subjects with NaNs
#         old_data = data
#         data_nan_indices = np.where(np.isin(ids, sub_ids_nan_vals))[0]
#         print(f"surf_nan_idx - {data_nan_indices}") # subjects with surf data that also have NaNs and need removal

#         # merge data and labels cause both have different subjects with nans
#         mask_data = np.ones(len(data), dtype=bool) # make mask of length of data == N, 
#         print(f"Data mask shape: {mask_data.shape}")
#         mask_data[data_nan_indices] = False # make where we found the NaNs ie subjects removed into false (all tohers are truw cause ones)
#         data_np = np.asarray(data) # make list to np array
#         data_nan_clean = data_np[mask_data] # that also allows us to index with the mask
#         print(f"Clean data shape, should be N - surf_nan_idx: {data_nan_clean.shape}")
#         # print(labels_nan_clean.shape)

#         # below is merging the two, so if surf needed subs removed then here we merge s.t. labels also now reflect the same and removed those subjs without surf
#         mask_surfaces = np.ones(labels_nan_clean.shape[0], dtype=bool)
#         mask_surfaces[data_nan_indices] = False # where the surf idx are for NaN make that false in label variable
#         merged_labels_with_surf = labels_nan_clean[mask_surfaces] # now that those are false, index w mask and we have new labels which is clean of own NaNs and surf NaNs
#         data = data_nan_clean
#         labels = merged_labels_with_surf
#         print(f"Cleaned surfaces and vec_netmats, make sure are same size and merged correctly: \nSURFACES:{data.shape} \nVEC_NETMATS:{labels.shape}")


In [ ]:
# num_subjects = data.shape[0]

# normalised_data = data
# print(normalised_data.shape)
# indices_mesh_triangles = pd.read_csv('../utils/triangle_indices_ico_{}_sub_ico_{}.csv'.format(ico,sub_ico))
# data = np.zeros((num_subjects, num_channels, num_patches, num_vertices))
# print(f'ICO-{sub_ico} dummy data shape: {data.shape}')

In [ ]:
# print(ids.shape)
# new_ids = ids[mask_surfaces]
# print(new_ids.shape)